# T2 — Reconhecimento de Produtos por Correspondência de Características Locais

Pipeline 100% clássico com OpenCV (SIFT + FLANN/BFMatcher). Sem OCR, sem redes neurais,
conforme `enunciado-t2.md`.

**Fluxo:**
1. Caminhos e imports
2. Configurações (`CFG`)
3. Carregar templates (`crops-t1/imagem-gold/`)
4. Funções do pipeline (extração, matching, classificação)
5. Teste em 1 segmento — visualização dos matches
6. Rodar sobre o dataset completo (`crops-t1/`)
7. Avaliação — matriz de confusão e acurácia no limiar escolhido
8. Calibração — varredura do único parâmetro exigido pelo enunciado (`min_match_frac`)
9. Exportar resultados (CSV) para o relatório técnico

## 1. Caminhos e imports
Este notebook espera rodar de dentro da pasta `t2/`, com `crops-t1/` (saída do T1, já corrigida
manualmente) e `crops-t1/imagem-gold/` (templates, um por classe) como subpastas.

In [ ]:
%pip install -q opencv-python numpy matplotlib

In [ ]:
import os
import glob
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

cv2.setRNGSeed(0)  # reprodutibilidade do RANSAC (match_fraction)

BASE_DIR      = os.path.dirname(os.path.abspath('__file__'))
SEGMENTOS_DIR = os.path.join(BASE_DIR, 'crops-t1')
TEMPLATES_DIR = os.path.join(SEGMENTOS_DIR, 'imagem-gold')

# nome do arquivo de template -> classe (pasta) correspondente em crops-t1/
TEMPLATE_MAP = {
    'meio-da-assa-gold.jpg':               '93000005_Meio_das_Asas_Congelado',
    'coxinha-das-asas-gold.jpg':           '93000006_Coxinhas_das_Asas_Congelado',
    'file-de-peito-gold.jpg':              '93000025_File_de_Peito_Congelado',
    'coracao-gold.jpg':                    '93000064_Coracao',
    'moela-gold.jpg':                      '93000068_Moela_Congelada',
    'file_de_coxas_e_sobrecoxas-gold.jpg': '93000096_File_de_Coxas_e_Sobre_Coxas_com_Pele_Congelado',
    'coxas_e_sobrecoxas-gold.jpg':         '93000106_Coxas_e_Sobrecoxas_Congelado',
}

print('Segmentos :', SEGMENTOS_DIR)
print('Templates :', TEMPLATES_DIR)
print('Classes   :', len(TEMPLATE_MAP))

## 2. Configurações
`min_match_frac` é o **único parâmetro de calibração exigido pelo enunciado**: a porcentagem
de correspondências válidas (inliers geométricos) sobre o total de pontos de interesse do
template. Os demais parâmetros (razão de Lowe, RANSAC) são fixos e compartilhados entre
todas as classes.

In [ ]:
CFG = {
    # Descritor local: 'SIFT' ou 'ORB' (ambos permitidos pelo enunciado)
    "descritor": "SIFT",
    "orb_n_features": 1000,

    # Teste de razão de Lowe — descarta matches ambíguos (2º vizinho quase tão bom quanto o 1º)
    "ratio_test": 0.75,

    # RANSAC (verificação geométrica) — só conta como match "de verdade" o que concorda com
    # uma única transformação afim. Filtra matches espúrios em texto repetido entre rótulos
    # (marca, "CONGELADO", tabela de peso) — principal fonte de falso positivo nesse método.
    "ransac_min_matches": 6,
    "ransac_residual_px": 8.0,

    # Abaixo disso, a imagem não tem evidência suficiente para extrair features úteis
    "min_keypoints": 8,

    # ÚNICO PARÂMETRO DE CALIBRAÇÃO (enunciado): % mínima de correspondências válidas
    # (inliers / keypoints do template) para aceitar a classificação. Abaixo disso -> "unknown".
    "min_match_frac": 0.08,
}

## 3. Funções do pipeline

In [ ]:
def criar_detector(cfg):
    if cfg['descritor'] == 'SIFT':
        return cv2.SIFT_create()
    return cv2.ORB_create(nfeatures=cfg['orb_n_features'])


def extrair_features(img_bgr, cfg):
    """Detecta pontos de interesse e extrai descritores locais (SIFT/ORB)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr
    det = criar_detector(cfg)
    kp, des = det.detectAndCompute(gray, None)
    if des is None or len(kp) < cfg['min_keypoints']:
        return None
    return kp, des


def _matcher(cfg):
    if cfg['descritor'] == 'SIFT':
        index_params  = dict(algorithm=1, trees=5)   # FLANN_INDEX_KDTREE
        search_params = dict(checks=50)
    else:
        index_params  = dict(algorithm=6, table_number=6, key_size=12, multi_probe_level=1)  # FLANN_INDEX_LSH
        search_params = dict(checks=50)
    return cv2.FlannBasedMatcher(index_params, search_params)


def _ratio_matches(matcher, des_a, des_b, cfg):
    """Casamento com teste de razão de Lowe: des_a (query) -> des_b (train)."""
    if len(des_a) < 2 or len(des_b) < 2:
        return {}
    pares = matcher.knnMatch(des_a, des_b, k=2)
    boas = {}
    for par in pares:
        if len(par) != 2:
            continue
        m, n = par
        if m.distance < cfg['ratio_test'] * n.distance:
            boas[m.queryIdx] = m.trainIdx
    return boas


def match_fraction(template_feats, seg_feats, cfg):
    """
    Casa template x segmento: razão de Lowe (nos dois sentidos, cross-check) + RANSAC afim.
    Score = correspondências válidas (inliers) / nº de keypoints do TEMPLATE.
    Retorna (score, matches_mutuos, kp_t, kp_s) para permitir visualização.
    """
    kp_t, des_t = template_feats
    kp_s, des_s = seg_feats
    matcher = _matcher(cfg)

    ab = _ratio_matches(matcher, des_t, des_s, cfg)   # template -> segmento
    ba = _ratio_matches(matcher, des_s, des_t, cfg)   # segmento -> template (cross-check)
    mutuos = [(t, s) for t, s in ab.items() if ba.get(s) == t]

    if len(mutuos) < cfg['ransac_min_matches']:
        return 0.0, mutuos, kp_t, kp_s

    src = np.float32([kp_t[t].pt for t, _ in mutuos])
    dst = np.float32([kp_s[s].pt for _, s in mutuos])
    _, inliers = cv2.estimateAffinePartial2D(
        src, dst, method=cv2.RANSAC, ransacReprojThreshold=cfg['ransac_residual_px'])

    if inliers is None:
        return 0.0, mutuos, kp_t, kp_s

    n_inliers = int(inliers.sum())
    score = n_inliers / len(kp_t)
    mutuos_inliers = [m for m, ok in zip(mutuos, inliers.ravel()) if ok]
    return score, mutuos_inliers, kp_t, kp_s


def classificar_segmento(seg_feats, templates_feats, cfg):
    """Compara o segmento contra TODOS os templates; vence o de maior score,
    desde que >= min_match_frac. Caso contrário -> None (não identificado)."""
    scores = {}
    for classe, tf in templates_feats.items():
        score, _, _, _ = match_fraction(tf, seg_feats, cfg)
        scores[classe] = score
    melhor_classe = max(scores, key=scores.get)
    melhor_score  = scores[melhor_classe]
    if melhor_score < cfg['min_match_frac']:
        return None, melhor_score, scores
    return melhor_classe, melhor_score, scores


print('Funcoes carregadas com sucesso!')

## 4. Carregar templates
Um template por classe, lido de `crops-t1/imagem-gold/`. Se algum arquivo do `TEMPLATE_MAP`
não existir, um aviso é impresso (a classe fica sem template e não pode ser reconhecida).

In [ ]:
templates_feats = {}
for nome_arquivo, classe in TEMPLATE_MAP.items():
    caminho = os.path.join(TEMPLATES_DIR, nome_arquivo)
    if not os.path.isfile(caminho):
        print('[AVISO] Template não encontrado:', caminho)
        continue
    img = cv2.imread(caminho)
    feats = extrair_features(img, CFG)
    if feats is None:
        print('[AVISO] Template sem keypoints suficientes:', nome_arquivo)
        continue
    templates_feats[classe] = feats
    print(f'{classe:60s} kp={len(feats[0]):4d}  ({nome_arquivo})')

print()
print('Templates carregados:', len(templates_feats), '/', len(TEMPLATE_MAP))

## 5. Teste em 1 segmento — visualização dos matches
Mostra os matches válidos (pós ratio-test + cross-check + RANSAC) entre um segmento e o
template da classe correta.

In [ ]:
CLASSE_TESTE = '93000064_Coracao'
pasta_classe = os.path.join(SEGMENTOS_DIR, CLASSE_TESTE)
arquivos_teste = sorted(glob.glob(os.path.join(pasta_classe, '*.jpg')) +
                         glob.glob(os.path.join(pasta_classe, '*.png')))

img_seg = cv2.imread(arquivos_teste[0])
seg_feats = extrair_features(img_seg, CFG)

pred, score, scores = classificar_segmento(seg_feats, templates_feats, CFG)
print('Arquivo   :', os.path.basename(arquivos_teste[0]))
print('Classe real:', CLASSE_TESTE)
print('Predição   :', pred, f'(score={score:.3f})')
print()
for classe, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f'  {classe:60s} {s:.3f}')

nome_template = [k for k, v in TEMPLATE_MAP.items() if v == CLASSE_TESTE][0]
img_tpl = cv2.imread(os.path.join(TEMPLATES_DIR, nome_template))
_, mutuos, kp_t, kp_s = match_fraction(templates_feats[CLASSE_TESTE], seg_feats, CFG)

dmatches = [cv2.DMatch(_queryIdx=t, _trainIdx=s, _imgIdx=0, _distance=0) for t, s in mutuos]
vis = cv2.drawMatches(img_tpl, kp_t, img_seg, kp_s, dmatches, None,
                       flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Matches válidos (inliers RANSAC): {len(mutuos)}  |  score={score:.3f}')
plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Rodar sobre o dataset completo (`crops-t1/`)
Classifica cada segmento de cada classe contra todos os templates. Guarda o resultado bruto
(scores contra todos os templates) para permitir recalibrar `min_match_frac` sem reprocessar.

In [ ]:
EXTENSOES = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
resultados = []  # (classe_real, nome_arquivo, scores_dict)

for classe_real in TEMPLATE_MAP.values():
    pasta = os.path.join(SEGMENTOS_DIR, classe_real)
    if not os.path.isdir(pasta):
        print('[AVISO] Pasta não encontrada:', pasta)
        continue
    arquivos = []
    for ext in EXTENSOES:
        arquivos.extend(glob.glob(os.path.join(pasta, ext)))
    arquivos = sorted(set(arquivos))

    for caminho in arquivos:
        nome = os.path.basename(caminho)
        img = cv2.imread(caminho)
        feats = extrair_features(img, CFG) if img is not None else None
        if feats is None:
            resultados.append((classe_real, nome, {}))
            continue
        _, _, scores = classificar_segmento(feats, templates_feats, CFG)
        resultados.append((classe_real, nome, scores))

    print('[CLASSE]', classe_real, '-', len(arquivos), 'segmentos processados')

print()
print('Total de segmentos avaliados:', len(resultados))

## 7. Avaliação — matriz de confusão e acurácia no limiar escolhido

In [ ]:
def avaliar(resultados, thresh):
    classes = sorted(set(TEMPLATE_MAP.values()))
    confusao = {c: {c2: 0 for c2 in classes + ['unknown']} for c in classes}
    acertos = falsos_positivos = unknown = 0

    for classe_real, nome, scores in resultados:
        if not scores:
            confusao[classe_real]['unknown'] += 1
            unknown += 1
            continue
        melhor_classe = max(scores, key=scores.get)
        melhor_score  = scores[melhor_classe]
        pred = melhor_classe if melhor_score >= thresh else 'unknown'
        confusao[classe_real][pred] += 1
        if pred == 'unknown':
            unknown += 1
        elif pred == classe_real:
            acertos += 1
        else:
            falsos_positivos += 1

    total = len(resultados)
    return {
        'acuracia':     acertos / total * 100,
        'unknown':      unknown,
        'falsos_pos':   falsos_positivos,
        'fp_rate':      falsos_positivos / total * 100,
        'confusao':     confusao,
        'total':        total,
    }


metricas = avaliar(resultados, CFG['min_match_frac'])
print(f"min_match_frac = {CFG['min_match_frac']}")
print(f"Acurácia        : {metricas['acuracia']:.1f}%")
print(f"Não identificado: {metricas['unknown']} / {metricas['total']}")
print(f"Falsos positivos: {metricas['falsos_pos']} / {metricas['total']}  ({metricas['fp_rate']:.1f}%)")
print()

classes = sorted(set(TEMPLATE_MAP.values()))
header = 'Real \\ Predito'.ljust(58) + ''.join(c[:6].rjust(8) for c in classes) + '  unknown'
print(header)
for c in classes:
    linha = c.ljust(58) + ''.join(str(metricas['confusao'][c][c2]).rjust(8) for c2 in classes)
    linha += str(metricas['confusao'][c]['unknown']).rjust(9)
    print(linha)

In [ ]:
def plotar_matriz_confusao(metricas, template_map):
    classes = sorted(set(template_map.values()))
    rotulos = classes + ['unknown']
    n = len(classes)

    matriz = np.zeros((n, len(rotulos)))
    for i, real in enumerate(classes):
        for j, pred in enumerate(rotulos):
            matriz[i, j] = metricas['confusao'][real][pred]

    nomes_curtos = [c.split('_', 1)[1][:22] for c in classes]

    fig, ax = plt.subplots(figsize=(1.1 * len(rotulos) + 2, 0.7 * n + 2))
    im = ax.imshow(matriz, cmap='Blues')

    ax.set_xticks(range(len(rotulos)))
    ax.set_xticklabels(nomes_curtos + ['unknown'], rotation=45, ha='right')
    ax.set_yticks(range(n))
    ax.set_yticklabels(nomes_curtos)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')
    ax.set_title(f"Matriz de confusão — min_match_frac={CFG['min_match_frac']}  "
                 f"(acurácia={metricas['acuracia']:.1f}%, FP={metricas['fp_rate']:.1f}%)")

    vmax = matriz.max()
    for i in range(n):
        for j in range(len(rotulos)):
            valor = int(matriz[i, j])
            cor = 'white' if valor > vmax * 0.5 else 'black'
            ax.text(j, i, str(valor), ha='center', va='center', color=cor, fontsize=9)

    fig.colorbar(im, ax=ax, label='nº de segmentos')
    plt.tight_layout()
    plt.show()


plotar_matriz_confusao(metricas, TEMPLATE_MAP)


## 8. Calibração — varredura de `min_match_frac`
Reaplica o limiar sobre os scores já calculados (sem reprocessar features). É o único
parâmetro de calibração exigido pelo enunciado; a tabela ajuda a escolher o ponto de operação
que minimiza falsos positivos sem descartar recall demais — conforme a prioridade do enunciado
("falso positivo pesa mais que detecção perdida").

In [ ]:
print(f'{"min_match_frac":>16} {"acuracia%":>10} {"unknown":>8} {"falsos_pos":>11} {"fp_rate%":>9}')
for t in [0.00, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]:
    m = avaliar(resultados, t)
    print(f'{t:16.2f} {m["acuracia"]:10.1f} {m["unknown"]:8d} {m["falsos_pos"]:11d} {m["fp_rate"]:9.1f}')

## 9. Exportar resultados (CSV) para o relatório técnico

In [ ]:
CSV_PATH = os.path.join(BASE_DIR, 'resultados_t2.csv')

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    classes = sorted(set(TEMPLATE_MAP.values()))
    writer.writerow(['classe_real', 'arquivo', 'predito', 'score_vencedor'] + classes)
    for classe_real, nome, scores in resultados:
        if not scores:
            writer.writerow([classe_real, nome, 'unknown', 0.0] + [0.0] * len(classes))
            continue
        melhor_classe = max(scores, key=scores.get)
        melhor_score  = scores[melhor_classe]
        pred = melhor_classe if melhor_score >= CFG['min_match_frac'] else 'unknown'
        writer.writerow([classe_real, nome, pred, round(melhor_score, 4)] +
                         [round(scores.get(c, 0.0), 4) for c in classes])

print('Resultados salvos em:', CSV_PATH)